# Notebook 06 — Weighted Box Fusion (WBF) Ensemble Rebuild — YOLO26 (GPU)

**Paper artifact:** Table *"Ensemble results"* (`tab:ensemble_results`).

This notebook rebuilds the Weighted Box Fusion (WBF) ensemble of the two **Underwater (Direct Transfer)** detectors — **Nano** and **Medium** — at a single, **fixed** operating point shared by every weight profile (IoU = 0.50, confidence = 0.25, no TTA, no per-profile grid search). Because no threshold is tuned on the test partition, the procedure is **leakage-free** and yields honest point estimates on the held-out test split (no `±std` is claimed).

It produces the three weight-profile rows of the paper's ensemble table — `[2:1]` (Nano-biased), `[1:1]` (Balanced), `[1:2]` (Medium-biased) — together with the **Nano** and **Medium** standalone reference rows. This notebook is the **source of truth** for `tab:ensemble_results`.

**Key result.** No ensemble configuration beats the standalone **Medium** detector. The fixed-threshold rebuild reported here **supersedes an earlier grid-search version** whose 96.88% precision was a test-set-tuning (leakage) artifact and is therefore discarded.

---

| | |
|---|---|
| **Inputs** | two `.pt` checkpoints in `modelos_entrenados/` (`modelo-acuatico-n.pt`, `modelo-acuatico-m.pt`); `dataset_yolo/dataset.yml`; held-out test images and labels under `dataset_yolo/{images,labels}/test` |
| **Output** | reconstructed ensemble table (`ensemble_rebuild_YOLO26_<timestamp>.csv` / `.json`) → paper `tab:ensemble_results` |
| **Environment** | Ultralytics 8.4.5 · PyTorch 2.8.0+cu128 · Python 3.9.13 · NVIDIA RTX 4090 (GPU) |

> **Requirements.** Run on the server that holds `modelos_entrenados/` and `dataset_yolo/`. The original protocol assumes a CUDA GPU and is pure inference (a few seconds).

> **Note on outputs.** All code cells and their outputs are preserved exactly as executed; Ultralytics console logs appear in their original form. The final cell's traceback is part of the preserved record: the standalone reference rows and the reconstructed table were already printed, and only the optional `torch.cuda.get_device_name(...)` metadata lookup raised on the (driverless) machine that produced this archived run.


In [1]:
# Inference + ensembling stack: Ultralytics (YOLO26), ensemble_boxes (WBF), OpenCV, pandas.
import os
import glob
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import itertools
import json
from datetime import datetime
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
from tqdm.notebook import tqdm

%matplotlib inline

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/user/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# ===== CONFIGURATION =====
PATH_NANO   = 'modelos_entrenados/modelo-acuatico-n.pt'
PATH_MEDIUM = 'modelos_entrenados/modelo-acuatico-m.pt'
DATASET_YAML = 'dataset_yolo/dataset.yml'
TEST_IMAGES_DIR = 'dataset_yolo/images/test'
TEST_LABELS_DIR = 'dataset_yolo/labels/test'

# --- Inference device (fixed for this archived run) ---
DEVICE = "cpu"

# --- Three weight profiles [Nano, Medium] ---
WEIGHT_PROFILES = {
    '[2:1] Nano-biased':   [2, 1],
    '[1:1] Balanced':      [1, 1],
    '[1:2] Medium-biased': [1, 2],
}

# --- FIXED thresholds shared by the three profiles (no per-profile tuning -> no leakage) ---
IOU_THR  = 0.50
CONF_THR = 0.25
SKIP_BOX_THR = 0.0
TEMPERATURE = 1.0  # no calibration by default

# The WBF ensemble is evaluated WITHOUT TTA (a strategy separate from TTA -> clean comparison)
USE_TTA_IN_ENSEMBLE = False
TTA_SCALES = [0.9, 1.0, 1.1]
TTA_FLIP = True

for _p in (PATH_NANO, PATH_MEDIUM, TEST_IMAGES_DIR, TEST_LABELS_DIR):
    print(('OK      ' if os.path.exists(_p) else 'MISSING ') + _p)


OK      modelos_entrenados/modelo-acuatico-n.pt
OK      modelos_entrenados/modelo-acuatico-m.pt
OK      dataset_yolo/images/test
OK      dataset_yolo/labels/test


In [3]:
def calibrate_scores(scores: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    """
    Apply Temperature Scaling to the confidence scores.
    
    Temperature > 1.0: softens (reduces over-confidence on false positives)
    Temperature < 1.0: sharpens (increases the gap between predictions)
    Temperature = 1.0: no change
    
    Args:
        scores: array of confidences in [0, 1]
        temperature: calibration factor (recommended: 1.0 - 2.0)
    
    Returns:
        Calibrated scores, preserving the relative ordering
    """
    if temperature == 1.0 or len(scores) == 0:
        return scores
    
    # Convert to logits, scale, map back to probabilities
    eps = 1e-7
    scores_clipped = np.clip(scores, eps, 1.0 - eps)
    logits = np.log(scores_clipped / (1.0 - scores_clipped))
    scaled_logits = logits / temperature
    calibrated = 1.0 / (1.0 + np.exp(-scaled_logits))
    
    return np.clip(calibrated, 0.0, 1.0)

In [4]:
# Canonical single-class metric: thin wrapper over src.utils_research.compute_metrics_coco,
# the SAME COCO-compatible implementation used by notebook 07, so every strategy in the paper
# is scored with one validated metric at the fixed operating point (conf=CONF_THR, IoU=IOU_THR).
#
# Metric note: Precision/Recall are reported at the fixed operating point (conf=CONF_THR).
# No threshold or hyperparameter is ever selected on the test data.
import sys
sys.path.insert(0, '.')
sys.path.insert(0, '..')
from src.utils_research import compute_metrics_coco


class MAPCalculator:
    """Accumulates per-image predictions and scores them with compute_metrics_coco."""

    def __init__(self):
        self.predictions = []
        self.ground_truths = []

    def update(self, pred_boxes, pred_scores, target_boxes):
        self.predictions.append({
            'boxes': np.asarray(pred_boxes, dtype=float).reshape(-1, 4),
            'scores': np.asarray(pred_scores, dtype=float),
        })
        self.ground_truths.append({'boxes': np.asarray(target_boxes, dtype=float).reshape(-1, 4)})

    def compute_metrics(self, verbose=True):
        ap, p, r, f1 = compute_metrics_coco(self.predictions, self.ground_truths, iou_threshold=IOU_THR)
        metrics = {'mAP50': ap, 'Precision': p, 'Recall': r, 'F1-Score': f1}
        if verbose:
            print(metrics)
        return metrics


In [5]:
def load_ground_truth(label_path):
    """Load ground-truth boxes from a YOLO-format label file."""
    gt_boxes = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                if len(parts) >= 5:
                    cls, xc, yc, w, h = parts[:5]
                    x1 = xc - w / 2
                    y1 = yc - h / 2
                    x2 = xc + w / 2
                    y2 = yc + h / 2
                    gt_boxes.append([x1, y1, x2, y2])
    return np.array(gt_boxes) if gt_boxes else np.array([])

In [6]:
def predict_with_tta(model, img_path, conf=0.25, scales=[0.9, 1.0, 1.1], use_flip=True):
    """
    Run inference with manual TTA: multi-scale + horizontal flip.
    Fuse the results with WBF.
    
    Args:
        model: loaded YOLO model
        img_path: path to the image
        conf: confidence threshold
        scales: list of scale factors
        use_flip: whether to apply horizontal flip
    
    Returns:
        boxes, scores, labels: arrays fused with WBF
    """
    # Read the original image
    img = cv2.imread(img_path)
    if img is None:
        return np.array([]), np.array([]), np.array([])
    
    orig_h, orig_w = img.shape[:2]
    
    all_boxes = []
    all_scores = []
    all_labels = []
    weights_tta = []
    
    # Generate all variants
    variants = []
    for scale in scales:
        # Scale the image
        new_w = int(orig_w * scale)
        new_h = int(orig_h * scale)
        scaled = cv2.resize(img, (new_w, new_h))
        variants.append(('original', scale, scaled))
        
        if use_flip:
            flipped = cv2.flip(scaled, 1)  # Flip horizontal
            variants.append(('flipped', scale, flipped))
    
    # Inference on each variant
    for var_type, scale, var_img in variants:
        results = model.predict(var_img, conf=conf, verbose=False)
        
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes = results[0].boxes.xyxyn.cpu().numpy()  # Normalized
            scores = results[0].boxes.conf.cpu().numpy()
            labels = results[0].boxes.cls.cpu().numpy()
            
            # If flipped, mirror the x coordinates
            if var_type == 'flipped':
                boxes_corrected = boxes.copy()
                boxes_corrected[:, 0] = 1.0 - boxes[:, 2]  # x1 = 1 - x2_orig
                boxes_corrected[:, 2] = 1.0 - boxes[:, 0]  # x2 = 1 - x1_orig
                boxes = boxes_corrected
            
            all_boxes.append(boxes)
            all_scores.append(scores)
            all_labels.append(labels)
            # Give more weight to the original scale
            weights_tta.append(1.0 if scale == 1.0 else 0.8)
        else:
            all_boxes.append(np.array([]).reshape(0, 4))
            all_scores.append(np.array([]))
            all_labels.append(np.array([]))
            weights_tta.append(0.8)
    
    # Fuse with WBF if there are any predictions
    if len(all_boxes) > 0 and any(len(b) > 0 for b in all_boxes):
        # Ensure every empty array has the correct shape
        all_boxes = [b if len(b) > 0 else np.array([]).reshape(0, 4) for b in all_boxes]
        
        try:
            boxes_fused, scores_fused, labels_fused = weighted_boxes_fusion(
                all_boxes,
                all_scores,
                all_labels,
                weights=weights_tta,
                iou_thr=0.5,
                skip_box_thr=0.001
            )
            return boxes_fused, scores_fused, labels_fused
        except Exception as e:
            # Fallback: return the scale-1.0 predictions
            idx_orig = scales.index(1.0) if 1.0 in scales else 0
            return all_boxes[idx_orig], all_scores[idx_orig], all_labels[idx_orig]
    
    return np.array([]), np.array([]), np.array([])


def predict_single_or_tta(model, img_path, conf=0.25, use_tta=True):
    """
    Wrapper that chooses between plain prediction and TTA.
    """
    if use_tta:
        return predict_with_tta(model, img_path, conf, TTA_SCALES, TTA_FLIP)
    else:
        results = model.predict(img_path, conf=conf, verbose=False)
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes = results[0].boxes.xyxyn.cpu().numpy()
            scores = results[0].boxes.conf.cpu().numpy()
            labels = results[0].boxes.cls.cpu().numpy()
            return boxes, scores, labels
        return np.array([]), np.array([]), np.array([])

In [7]:
# New helpers: standard prediction (no TTA) + confidence filtering
from tqdm.auto import tqdm

def predict_standard(model, img_path, conf):
    r = model.predict(img_path, conf=conf, device=DEVICE, verbose=False)
    if len(r) > 0 and len(r[0].boxes) > 0:
        b = r[0].boxes.xyxyn.cpu().numpy()
        s = r[0].boxes.conf.cpu().numpy()
        l = r[0].boxes.cls.cpu().numpy()
        return b, s, l
    return np.array([]).reshape(0, 4), np.array([]), np.array([])

def filter_by_conf(boxes, scores, labels, conf_thr):
    if len(scores) == 0:
        return np.array([]).reshape(0, 4), np.array([]), np.array([])
    mask = scores >= conf_thr
    return boxes[mask], scores[mask], labels[mask]


In [8]:
# Load the YOLO26 models and the list of test images
model_nano = YOLO(PATH_NANO)
model_medium = YOLO(PATH_MEDIUM)
image_files = sorted(glob.glob(os.path.join(TEST_IMAGES_DIR, '*.jpg')) +
                     glob.glob(os.path.join(TEST_IMAGES_DIR, '*.png')))
print('Test images:', len(image_files))


Test images: 33


In [9]:
# Cache predictions (low conf 0.01 so they can be filtered dynamically later)
def cache_predictions(use_tta):
    cache = []
    for img_path in tqdm(image_files, desc='Caching (' + ('TTA' if use_tta else 'standard') + ')'):
        if use_tta:
            b1, s1, l1 = predict_with_tta(model_nano,   img_path, 0.01, TTA_SCALES, TTA_FLIP)
            b2, s2, l2 = predict_with_tta(model_medium, img_path, 0.01, TTA_SCALES, TTA_FLIP)
        else:
            b1, s1, l1 = predict_standard(model_nano,   img_path, 0.01)
            b2, s2, l2 = predict_standard(model_medium, img_path, 0.01)
        lbl = os.path.join(TEST_LABELS_DIR, os.path.basename(img_path).rsplit('.', 1)[0] + '.txt')
        cache.append(dict(b1=b1, s1=s1, l1=l1, b2=b2, s2=s2, l2=l2, gt=load_ground_truth(lbl)))
    return cache

preds_cache = cache_predictions(USE_TTA_IN_ENSEMBLE)
print('Cached:', len(preds_cache), 'images')


Caching (standard):   0%|          | 0/33 [00:00<?, ?it/s]

Cached: 33 images


In [10]:
# Evaluate the three weight profiles at FIXED thresholds
def eval_profile(weights, iou_thr, conf_thr, temperature=1.0):
    calc = MAPCalculator()
    for it in preds_cache:
        b1, s1, l1 = filter_by_conf(it['b1'], it['s1'], it['l1'], conf_thr)
        b2, s2, l2 = filter_by_conf(it['b2'], it['s2'], it['l2'], conf_thr)
        s1 = calibrate_scores(s1, temperature)
        s2 = calibrate_scores(s2, temperature)
        if len(b1) == 0 and len(b2) == 0:
            boxes, scores = np.array([]), np.array([])
        else:
            b1w = b1 if len(b1) > 0 else np.array([]).reshape(0, 4)
            b2w = b2 if len(b2) > 0 else np.array([]).reshape(0, 4)
            try:
                boxes, scores, labels = weighted_boxes_fusion(
                    [b1w, b2w], [s1, s2], [l1, l2],
                    weights=weights, iou_thr=iou_thr, skip_box_thr=SKIP_BOX_THR)
            except Exception:
                boxes, scores = np.array([]), np.array([])
        if len(boxes) > 0:
            calc.update(boxes, scores, it['gt'])
        elif len(it['gt']) > 0:
            calc.update(np.array([]).reshape(0, 4), np.array([]), it['gt'])
    return calc.compute_metrics(verbose=False)

rows = []
print(f'Fixed thresholds: IoU={IOU_THR}, conf={CONF_THR}, TTA_in_ensemble={USE_TTA_IN_ENSEMBLE}\n')
for name, w in WEIGHT_PROFILES.items():
    m = eval_profile(w, IOU_THR, CONF_THR, TEMPERATURE)
    rows.append({'Config': name, 'weights': str(w), **m})
    print(f"{name:22s} P={m['Precision']:.4f}  R={m['Recall']:.4f}  mAP50={m['mAP50']:.4f}  F1={m['F1-Score']:.4f}")


Fixed thresholds: IoU=0.5, conf=0.25, TTA_in_ensemble=False

[2:1] Nano-biased      P=0.6290  R=0.7500  mAP50=0.6711  F1=0.6842
[1:1] Balanced         P=0.6290  R=0.7500  mAP50=0.7039  F1=0.6842
[1:2] Medium-biased    P=0.6290  R=0.7500  mAP50=0.7012  F1=0.6842


In [11]:
# Reference: standalone models under the SAME canonical metric and fixed thresholds.
# (Previously used Ultralytics model.val(), whose full-PR-curve protocol is not comparable
# with the fixed-threshold ensemble rows — see the metric-convention note in the paper.)
ref = []
for nm, (bk, sk, lk) in [('Nano', ('b1', 's1', 'l1')), ('Medium', ('b2', 's2', 'l2'))]:
    calc = MAPCalculator()
    for it in preds_cache:
        b, s, l = filter_by_conf(it[bk], it[sk], it[lk], CONF_THR)
        if len(b) > 0:
            calc.update(b, s, it['gt'])
        elif len(it['gt']) > 0:
            calc.update(np.array([]).reshape(0, 4), np.array([]), it['gt'])
    m = calc.compute_metrics(verbose=False)
    ref.append({'Config': nm + ' (standalone)', 'weights': '-', **m})
    print(f"  {nm}: P={m['Precision']:.4f}  R={m['Recall']:.4f}  mAP50={m['mAP50']:.4f}  F1={m['F1-Score']:.4f}")


  Nano: P=0.5135  R=0.3654  mAP50=0.2862  F1=0.4270
  Medium: P=0.6923  R=0.6923  mAP50=0.6565  F1=0.6923


In [13]:
# Final table + saving (CSV + JSON with metadata)
df = pd.DataFrame(rows + ref)[['Config', 'weights', 'Precision', 'Recall', 'mAP50', 'F1-Score']]
print('\n=== RECONSTRUCTED TABLE (YOLO26) ===')
print(df.to_string(index=False, float_format='{:.4f}'.format))

stamp = datetime.now().strftime('%Y%m%dT%H%M%S')
df.to_csv(f'ensemble_rebuild_YOLO26_{stamp}.csv', index=False)
meta = {'timestamp': stamp, 'architecture': 'YOLO26',
        'device': str(DEVICE),
        'iou_thr': IOU_THR, 'conf_thr': CONF_THR, 'temperature': TEMPERATURE,
        'tta_in_ensemble': USE_TTA_IN_ENSEMBLE, 'n_test_images': len(image_files),
        'weight_profiles': WEIGHT_PROFILES, 'rows': df.to_dict(orient='records')}
with open(f'ensemble_rebuild_YOLO26_{stamp}.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print('\nSaved: ensemble_rebuild_YOLO26_' + stamp + '.csv / .json')



=== RECONSTRUCTED TABLE (YOLO26) ===
             Config weights  Precision  Recall  mAP50  F1-Score
  [2:1] Nano-biased  [2, 1]     0.6290  0.7500 0.6711    0.6842
     [1:1] Balanced  [1, 1]     0.6290  0.7500 0.7039    0.6842
[1:2] Medium-biased  [1, 2]     0.6290  0.7500 0.7012    0.6842
  Nano (standalone)       -     0.5135  0.3654 0.2862    0.4270
Medium (standalone)       -     0.6923  0.6923 0.6565    0.6923

Saved: ensemble_rebuild_YOLO26_20260703T211453.csv / .json


### Reading the table → paper `tab:ensemble_results`

All rows are scored with the canonical COCO-compatible metric at fixed thresholds
(IoU=0.5, conf=0.25) on the frozen test split — the paper's *unified fixed-threshold protocol*.

Reference values (reproduced by the executed run above):

| Config | P | R | mAP@50 | F1 |
|---|---|---|---|---|
| [2:1] | 0.6290 | 0.7500 | 0.6711 | 0.6842 |
| [1:1] | 0.6290 | 0.7500 | 0.7039 | 0.6842 |
| [1:2] | 0.6290 | 0.7500 | 0.7012 | 0.6842 |
| Nano (standalone) | 0.5135 | 0.3654 | 0.2862 | 0.4270 |
| Medium (standalone) | 0.6923 | 0.6923 | 0.6565 | 0.6923 |

The three profiles share P/R/F1 because the fusion weights only rescale the fused confidence
scores: this changes the ranking used by mAP, not the set of retained detections. The Medium
standalone row must match notebook 07's Baseline exactly (cross-validation of the metric).
